## Следующие шаги

1. **Сопоставление спутниковых данных с видео** - реализация алгоритмов сопоставления признаков (SIFT, ORB, SURF)
2. **Уточнение геолокализации** - использование результатов сопоставления для повышения точности
3. **Интеграция телеметрических данных** - привязка данных с датчиков и GPS (если доступны)
4. **Создание 3D модели** - построение трехмерной карты на основе видеоданных

In [ ]:
# Анализ объектов на сегментированном изображении
def analyze_objects(frame, mask):
    \"\"\"Анализирует объекты на основе маски\"\"\"
    # Морфологические операции для улучшения маски
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_OPEN, kernel)
    
    # Поиск контуров
    contours, _ = cv2.findContours(mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    objects = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 100:  # Фильтр по минимальной площади
            x, y, w, h = cv2.boundingRect(cnt)
            objects.append({
                'x': x, 'y': y,
                'width': w, 'height': h,
                'area': area,
                'perimeter': cv2.arcLength(cnt, True)
            })
    
    return objects, mask_cleaned, contours

if video_files:
    cap = cv2.VideoCapture(str(video_files[0]))
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        print(f"🔍 Обнаружение объектов\\n")
        
        # Используем маску земли для анализа объектов
        _, ground_mask = segment_sky_ground(frame)
        objects, mask_cleaned, contours = analyze_objects(frame, ground_mask)
        
        print(f"Обнаружено объектов: {len(objects)}")
        
        if objects:
            print("\\nСписок объектов:")
            for i, obj in enumerate(objects[:5]):  # Показываем первые 5
                print(f"  {i+1}. Позиция: ({obj['x']}, {obj['y']}), "
                      f"Размер: {obj['width']}x{obj['height']}, "
                      f"Площадь: {obj['area']:.0f} px²")
        
        # Визуализация с рисованием контуров
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_with_contours = frame_rgb.copy()
        cv2.drawContours(frame_with_contours, contours, -1, (0, 255, 0), 2)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        axes[0].imshow(mask_cleaned, cmap='gray')
        axes[0].set_title('Очищенная маска')
        axes[0].axis('off')
        
        axes[1].imshow(frame_with_contours)
        axes[1].set_title(f'Обнаруженные объекты ({len(objects)} шт.)')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()

## 6. Обнаружение и анализ объектов

Используем контурный анализ и морфологические операции для выделения объектов.

In [ ]:
# Функции для сегментации изображений
def segment_by_color_range(frame, lower_hsv, upper_hsv):
    \"\"\"Сегментирует изображение по диапазону цветов в HSV\"\"\"
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower_hsv, upper_hsv)
    return mask

def segment_by_threshold(frame, threshold=127):
    \"\"\"Сегментирует изображение по порогу яркости\"\"\"
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    return binary

def segment_by_edges(frame, canny_low=50, canny_high=150):
    \"\"\"Сегментирует изображение по контурам (Canny)\"\"\"
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, canny_low, canny_high)
    return edges

def segment_sky_ground(frame):
    \"\"\"
    Сегментирует небо и землю (для аэрофотосъемки)
    Небо обычно имеет более высокое значение В в HSV
    \"\"\"
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Диапазон для неба (синие и голубые цвета)
    lower_sky = np.array([90, 50, 50])
    upper_sky = np.array([130, 255, 255])
    
    sky_mask = cv2.inRange(hsv, lower_sky, upper_sky)
    ground_mask = cv2.bitwise_not(sky_mask)
    
    return sky_mask, ground_mask

# Применяем сегментацию к первому кадру
if video_files:
    cap = cv2.VideoCapture(str(video_files[0]))
    ret, frame = cap.read()
    cap.release()
    
    if ret:
        print(f"🖼️  Сегментация кадра из {video_files[0].name}\\n")
        
        # Примяем различные методы сегментации
        edges = segment_by_edges(frame)
        sky_mask, ground_mask = segment_sky_ground(frame)
        
        # Визуализация результатов
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Исходный кадр
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[0, 0].imshow(frame_rgb)
        axes[0, 0].set_title('Исходный кадр')
        axes[0, 0].axis('off')
        
        # Края (Canny)
        axes[0, 1].imshow(edges, cmap='gray')
        axes[0, 1].set_title('Обнаруженные края (Canny)')
        axes[0, 1].axis('off')
        
        # Маска неба
        axes[1, 0].imshow(sky_mask, cmap='gray')
        axes[1, 0].set_title('Маска неба')
        axes[1, 0].axis('off')
        
        # Маска земли
        axes[1, 1].imshow(ground_mask, cmap='gray')
        axes[1, 1].set_title('Маска земли/объектов')
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print("✓ Сегментация завершена")

## 5. Сегментация изображений

Применяем методы сегментации для выделения ключевых объектов на кадрах видео.

In [ ]:
# Функция для геореференцирования кадров
def georeference_frames(video_path, start_lat, start_lon, start_height, sample_rate=30):
    \"\"\"
    Геореференцирует кадры видео.
    
    Args:
        video_path: путь к видеофайлу
        start_lat: начальная широта
        start_lon: начальная долгота
        start_height: начальная высота (метры)
        sample_rate: каждый N-ый кадр для анализа
    \"\"\"
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    frames_data = []
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Обработка каждого N-ого кадра
        if frame_idx % sample_rate == 0:
            # Время кадра в видео
            time_sec = frame_idx / fps
            
            # Примерная высота (может быть уточнена по телеметрии)
            current_height = start_height
            
            # Примерный расчет смещения координат
            # (для более точного расчета нужны данные с GPS/датчиков)
            frames_data.append({
                'frame_id': frame_idx,
                'time_sec': time_sec,
                'latitude': start_lat,
                'longitude': start_lon,
                'height_m': current_height,
                'frame_shape': frame.shape
            })
        
        frame_idx += 1
    
    cap.release()
    return pd.DataFrame(frames_data)

# Геореференцирование кадров из первого видеофайла
if video_files:
    print(f"🗺️  Геореференцирование видео: {video_files[0].name}\\n")
    frames_df = georeference_frames(video_files[0], START_LAT, START_LON, START_HEIGHT, sample_rate=30)
    
    print(f"Обработано {len(frames_df)} кадров:")
    print(frames_df.head(10))
    print(f"\\n📊 Статистика:")
    print(f"  Всего кадров для анализа: {len(frames_df)}")
    print(f"  Диапазон времени: {frames_df['time_sec'].min():.2f} - {frames_df['time_sec'].max():.2f} сек")

## 4. Геореференцирование видеокадров

Связываем кадры видео с геокоординатами, используя начальные координаты и данные высоты.

In [ ]:
# Параметры проекта
START_LAT = 55.086025
START_LON = 38.149033
START_HEIGHT = 750  # метры

# Создание интерактивной карты со спутниковым слоем
print(f"📍 Создание карты для координат: {START_LAT}, {START_LON}")
print(f"   Высота при отцепе: {START_HEIGHT} м\\n")

# Основная карта со спутниковым слоем
m = folium.Map(
    location=[START_LAT, START_LON],
    zoom_start=15,
    tiles='OpenStreetMap'
)

# Добавляем спутниковый слой
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri',
    name='Satellite (Esri)',
    overlay=False,
    control=True
).add_to(m)

# Добавляем точку старта
folium.Marker(
    location=[START_LAT, START_LON],
    popup=f'Начало полета<br>Высота: {START_HEIGHT}м',
    tooltip='Точка старта',
    icon=folium.Icon(color='green', icon='info-sign')
).add_to(m)

# Добавляем окружность, соответствующую примерной дальности видеокамеры
# при высоте 750м (примерная зона видимости)
folium.Circle(
    location=[START_LAT, START_LON],
    radius=500,  # метры
    popup='Примерная зона видимости (500м)',
    color='blue',
    fill=True,
    fillColor='blue',
    fillOpacity=0.1
).add_to(m)

# Добавляем управление слоями
folium.LayerControl().add_to(m)

# Сохраняем карту
map_path = RESULTS_DIR / 'satellite_map.html'
m.save(str(map_path))
print(f"✓ Карта сохранена: {map_path}")

# Отображаем карту
m

## 3. Загрузка спутниковых карт

Загружаем спутниковые снимки для заданной геолокации (55.086025, 38.149033) с помощью Folium и создаем интерактивную карту.

In [ ]:
# Функция для извлечения метаданных с помощью ffprobe
def extract_metadata_ffprobe(video_path):
    \"\"\"Извлекает метаданные из видеофайла с помощью ffprobe\"\"\"
    try:
        cmd = [
            'ffprobe',
            '-v', 'quiet',
            '-print_format', 'json',
            '-show_format',
            '-show_streams',
            str(video_path)
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return json.loads(result.stdout)
    except Exception as e:
        print(f"Ошибка при извлечении метаданных: {e}")
    return None

# Функция для красивого вывода метаданных
def print_metadata(metadata, video_name):
    \"\"\"Выводит метаданные в удобном формате\"\"\"
    print(f"\\n{'='*60}")
    print(f"Метаданные: {video_name}")
    print(f"{'='*60}")
    
    if not metadata:
        print("❌ Метаданные не найдены")
        return
    
    # Информация о формате
    if 'format' in metadata:
        fmt = metadata['format']
        print(f"\\n📋 Информация о формате:")
        print(f"  Формат: {fmt.get('format_name', 'N/A')}")
        print(f"  Продолжительность: {float(fmt.get('duration', 0)):.2f} сек")
        print(f"  Размер: {int(fmt.get('size', 0)) / (1024**3):.2f} GB")
        print(f"  Битрейт: {int(fmt.get('bit_rate', 0)) / 1000000:.2f} Mbps")
        
        # Теги (включая GoPro метаданные)
        if 'tags' in fmt:
            print(f"\\n🏷️  Теги метаданных:")
            for key, value in fmt['tags'].items():
                if value:
                    print(f"  {key}: {value}")
    
    # Информация о потоках
    if 'streams' in metadata:
        print(f"\\n🎬 Информация о видеопотоках:")
        for i, stream in enumerate(metadata['streams']):
            if stream.get('codec_type') == 'video':
                print(f"\\n  Видеопоток {i}:")
                print(f"    Кодек: {stream.get('codec_name', 'N/A')}")
                print(f"    Разрешение: {stream.get('width')}x{stream.get('height')}")
                print(f"    FPS: {eval(stream.get('avg_frame_rate', '0/1'))[0] if '/' in stream.get('avg_frame_rate', '0/1') else 'N/A'}")
                
                # Проверка на наличие сторон метаданных (side_data)
                if 'side_data_list' in stream:
                    print(f"    Side Data: {len(stream['side_data_list'])} записей найдено")
                    for side_data in stream['side_data_list']:
                        print(f"      - {side_data.get('side_data_type', 'Unknown')}")

# Анализ метаданных для всех видеофайлов
for video_file in sorted(video_files):
    metadata = extract_metadata_ffprobe(video_file)
    print_metadata(metadata, video_file.name)

## 2. Извлечение метаданных из видео GoPro

Используем ffprobe для извлечения подробных метаданных из видеофайла (GPS, датчики, телеметрия).

In [ ]:
# Отображение первого кадра из первого видеофайла
if video_files:
    cap = cv2.VideoCapture(str(video_files[0]))
    ret, first_frame = cap.read()
    cap.release()
    
    if ret:
        fig, ax = plt.subplots(figsize=(12, 8))
        # Конвертируем BGR в RGB для правильного отображения
        frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame_rgb)
        ax.set_title(f'Первый кадр: {video_files[0].name}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print(f"✓ Первый кадр видео загружен: {first_frame.shape}")

In [ ]:
# Поиск видеофайлов
video_files = list(Path('/Users/kirill/Documents/Avia-Geo-Fusion').glob('*.MP4'))
print(f"Найденные видеофайлы: {[f.name for f in video_files]}\n")

# Функция для анализа видео
def analyze_video(video_path):
    """Анализирует видеофайл и извлекает основную информацию"""
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        return None
    
    # Получение основной информации
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    info = {
        'filename': video_path.name,
        'fps': fps,
        'frame_count': frame_count,
        'resolution': f'{width}x{height}',
        'duration_sec': duration,
        'duration_min': duration / 60 if duration > 0 else 0
    }
    
    # Извлечение первого кадра
    ret, first_frame = cap.read()
    cap.release()
    
    return info, first_frame if ret else None

# Анализ всех видеофайлов
video_info_list = []
for video_file in sorted(video_files):
    info, first_frame = analyze_video(video_file)
    if info:
        video_info_list.append(info)
        print(f"📹 {info['filename']}")
        print(f"   Разрешение: {info['resolution']}")
        print(f"   FPS: {info['fps']:.2f}")
        print(f"   Кадров: {info['frame_count']}")
        print(f"   Длительность: {info['duration_min']:.2f} мин ({info['duration_sec']:.1f} сек)")
        print()

## 1. Загрузка и анализ видео с GoPro

Загружаем видеофайл, извлекаем основную информацию о видео и первые кадры.

In [ ]:
# Импорт необходимых библиотек
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess
import json
from datetime import datetime

# Для работы с метаданными видео
try:
    from PIL import Image
    from PIL.ExifTags import TAGS
except ImportError:
    print("PIL not installed, will install if needed")

# Визуализация
import folium
from folium import plugins

# Установка путей к данным
PROJECT_ROOT = Path('/Users/kirill/Documents/Avia-Geo-Fusion')
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'

print("✓ Библиотеки загружены успешно")

# Совмещение спутниковых карт и видеоданных для геолокализации в авиации

**Проект:** Уточнение геолокализации в задачах гражданской авиации путем совмещения спутниковых карт и данных с бортовой видеокамеры

**Основные параметры:**
- Высота при отцепе (взлете): 750 м
- Начальные координаты: 55.086025, 38.149033
- Источник видео: GoPro (GOPR0269.MP4, GP010269.MP4)